In [3]:
!pip uninstall playwright-stealth -y
!pip install playwright-stealth==1.0.6

Found existing installation: playwright-stealth 2.0.3
Uninstalling playwright-stealth-2.0.3:
  Successfully uninstalled playwright-stealth-2.0.3

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
!pip install DrissionPage 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 2.5 MB/s  0:00:03m 2.5 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [DrissionPage]0m 6/7 [DrissionPage]

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from DrissionPage import ChromiumPage, ChromiumOptions
import time

co = ChromiumOptions()
co.set_argument("--no-sandbox")
co.set_argument("--lang=fr-FR")

page = ChromiumPage(addr_or_opts=co)
page.get("https://www.tripadvisor.fr/Hotel_Review-g293736-d299645-Reviews-Ibis_Rabat_Agdal-Rabat_Rabat_Sale_Kenitra.html")

time.sleep(4)  # attendre le chargement

# Scroll pour charger les avis
for _ in range(8):
    page.scroll.down(700)
    time.sleep(0.3)

time.sleep(2)

# ── Sauvegarde le HTML complet ──
with open("tripadvisor_debug.html", "w", encoding="utf-8") as f:
    f.write(page.html)

print("✅ HTML sauvegardé dans tripadvisor_debug.html")
print(f"   Taille : {len(page.html)} caractères")

# ── Cherche les divs qui contiennent des avis ──
print("\n── Classes des divs principaux ──")
divs = page.eles("css:div[class]")
classes_vues = set()
for d in divs[:200]:
    cls = d.attr("class") or ""
    if any(kw in cls.lower() for kw in ["review", "avis", "list", "card", "item", "rating"]):
        if cls not in classes_vues:
            print(f"  {cls[:80]}")
            classes_vues.add(cls)

page.quit()

✅ HTML sauvegardé dans tripadvisor_debug.html
   Taille : 1285387 caractères

── Classes des divs principaux ──


In [7]:
"""
TripAdvisor Hotel Reviews Scraper — FINAL
==========================================
Librairie  : DrissionPage
Sélecteurs : validés sur HTML réel (mai 2026)
Cible      : Ibis Rabat Agdal

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, json, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = (
    "https://www.tripadvisor.fr/Hotel_Review-g293736-d299645-Reviews-"
    "Ibis_Rabat_Agdal-Rabat_Rabat_Sale_Kenitra.html"
)
MAX_PAGES   = None          # None = toutes les pages, ex: 5 pour limiter
HEADLESS    = False
OUTPUT_CSV  = "tripadvisor_reviews.csv"
OUTPUT_JSON = "tripadvisor_reviews.json"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    """'4 sur 5 bulles' → 4.0"""
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    """'Wab R a écrit un avis en avr. 2026' → 'avr. 2026'"""
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    btns = page.eles('css:button.UikNM')
    for btn in btns:
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # ── Auteur ──────────────────────────────────────────────────
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # ── Date de publication ──────────────────────────────────────
        # div.biGQs.VImYz.AWdfh  →  "Wab R a écrit un avis en avr. 2026"
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # ── Contributions ────────────────────────────────────────────
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # ── Note ─────────────────────────────────────────────────────
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # ── Titre ─────────────────────────────────────────────────────
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # ── Texte de l'avis ───────────────────────────────────────────
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # ── Date séjour & Type de voyage ──────────────────────────────
        # div.ERnxS contient les deux span.xENVe
        # Premier span.xENVe = date séjour, Second = type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour  = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage  = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination  —  a[aria-label="Page suivante"]
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        # Sélecteur validé depuis le HTML réel
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde
# ─────────────────────────────────────────
def save(all_reviews: list):
    dicts = [asdict(r) for r in all_reviews]
    df = pd.DataFrame(dicts)
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(dicts, f, ensure_ascii=False, indent=2)
    print(f"✅ JSON → {OUTPUT_JSON}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — FINAL")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            # Scroll progressif pour déclencher le lazy-load
            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde des données collectées...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — FINAL
   URL       : https://www.tripadvisor.fr/Hotel_Review-g293736-d299645-Reviews-Ibis_Rabat_Agdal-Rabat_Rabat_Sale_Kenitra.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 120

📄 Page 13
  → 10 carte(s)
  ✓ 10 avis  |  t

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Wab R,3 contributions,avr. 2026,4.0,Bon séjour,Très bon séjour à l’hôtel ibis Rabat. Chambre ...,avril 2026,A voyagé en famille
1,OnAir28678313555,1 contribution,avr. 2026,5.0,Beautifull resepsionis,Good hotel dan beautifull 💙💙,avril 2026,A voyagé pour affaires
2,Anthony B,1 contribution,avr. 2026,5.0,Excellent.,"Très bonnes chambres, nourriture, soins et pré...",avril 2026,A voyagé avec des amis
3,Ngone F,1 contribution,avr. 2026,5.0,Decouverte de l'hôtel ibis,'nous somme arrivé a hôtel ibis depuis hier l'...,avril 2026,A voyagé pour affaires
4,Ratiba S,1 contribution,avr. 2026,5.0,Hôtel agréable et reposant,"Un très bel accueil. Des employés aimables, so...",avril 2026,A voyagé pour affaires
...,...,...,...,...,...,...,...,...
667,billyfisher,Manchester,août 2007,1.0,Ibis Horribilis,"Un hôtel à fuir. A notre arrivée, il a fallu r...",juillet 2007,A voyagé en couple
668,sanditaylor,"Londres, Royaume-Uni",nov. 2006,3.0,Pratique et basique,Cet hôtel correspond tout à fait à ce qu’on pe...,novembre 2006,A voyagé en couple
669,gokiwi,"Christchurch, Nouvelle-Zélande",oct. 2006,2.0,Le Ramadan mort,Nous avons passé deux nuits ici avec ma femme ...,septembre 2006,
670,Inspire21563,"Brussels, Belgium",juil. 2006,1.0,Ne faites pas de réservation par Internet à l'...,Faites attention ! J’avais réservé une nuit à ...,juin 2006,


In [10]:
df = pd.read_csv("tripadvisor_reviews.csv")
df.shape

(672, 8)

In [2]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Oujda
================================================
Librairie  : DrissionPage
Cible      : Ibis Oujda - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g298351-d299644-Reviews-Ibis_Oujda-Oujda_Oriental.html"
MAX_PAGES  = None          # None = toutes les pages
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_oujda.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV uniquement
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Oujda")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Oujda
   URL       : https://www.tripadvisor.fr/Hotel_Review-g298351-d299644-Reviews-Ibis_Oujda-Oujda_Oriental.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 120

📄 Page 13
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Younas A,2 contributions,janv. 2026,5.0,Un bonne ibis au maroc,"L'hôtel est fantastique tout simplement , le p...",janvier 2026,A voyagé avec des amis
1,elkouch h,2 contributions,janv. 2026,5.0,la satisfaction10/10,Excellent service comme d'habitude a toutes l'...,janvier 2026,A voyagé avec des amis
2,Hossin E,2 contributions,janv. 2026,5.0,Merci ibis oujda,Excellent sejour a ibis oujda et un grand Merc...,janvier 2026,A voyagé en famille
3,Hassan B,2 contributions,janv. 2026,5.0,Ibis oujda ♥️♥️,personnel a la hauteur accueil chaleureux je r...,janvier 2026,A voyagé pour affaires
4,Mohemad A,2 contributions,janv. 2026,5.0,Merci Ibis oujda,"Très bon séjour, cuisine de qualité bravo au C...",janvier 2026,A voyagé en couple
...,...,...,...,...,...,...,...,...
268,Extraordinary281735,casablanca,févr. 2011,3.0,Gare et centre ville à proximité,Hôtel adjacent à la gare.\nA 200-300 mètres du...,janvier 2011,A voyagé pour affaires
269,Voyage271579,hyeres,nov. 2010,3.0,Avec enfant s'abstenir...,Hotel typique ibis. Sauf que venir en couple a...,octobre 2010,A voyagé en famille
270,must06,cagnes sur mer,sept. 2010,1.0,lamentable,grosses lacqune professionnels et attitude ins...,septembre 2010,A voyagé pour affaires
271,leila6,"Fribourg, Suisse",août 2010,4.0,sympa,"Hôtel très agréable, personnel d'accueil très ...",juillet 2010,A voyagé pour affaires


In [5]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Fès
=============================================
Librairie  : DrissionPage
Cible      : Ibis Fes - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g293733-d299642-Reviews-Ibis_Fes-Fez_Fez_Meknes.html"
MAX_PAGES  = None          # None = toutes les pages
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_fes.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Fès")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Fès
   URL       : https://www.tripadvisor.fr/Hotel_Review-g293733-d299642-Reviews-Ibis_Fes-Fez_Fez_Meknes.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre toutes langues via URL

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 120

📄 Page 13
  → 10 carte(s

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Venture11064191903,1 contribution,nov. 2025,5.0,Excellence,"Propreté,calme et accueil chaleureux.Je me sui...",novembre 2025,A voyagé en solo
1,Patricia H,"Manchester, Royaume-Uni",nov. 2025,4.0,Budget amical Oasis du centre-ville juste une ...,Nous avons passé trois nuits dans cet emplacem...,novembre 2025,A voyagé en couple
2,Excursion16727753481,3 contributions,sept. 2025,2.0,Décevant et insupportable d'année en année,"Malgré avoir réservé une chambre double, on no...",septembre 2025,A voyagé en couple
3,Traveler00106826122,1 contribution,août 2025,1.0,Service inacceptable et manque d'hébergement,"À l'arrivée, nous avons demandé un mini-réfrig...",août 2025,A voyagé en famille
4,Chris M,1 contribution,mai 2025,5.0,Parfait,"Hôtel très propre, parc avec piscine très agré...",mai 2025,A voyagé en couple
...,...,...,...,...,...,...,...,...
1136,Al M,"Duenweg, Missouri",janv. 2007,5.0,Un excellent rapport qualité/prix à Fez,J'ai réservé cet hôtel après avoir passé un ex...,novembre 2006,A voyagé pour affaires
1137,Heike65,"Seattle, Etat de Washington",oct. 2006,4.0,Un bon rapport qualité/prix et un délicieux pe...,Situé au bord de la rue et à une courte distan...,mai 2004,
1138,elsobranty,canada,août 2006,4.0,Super pour une réduction treavellers quand mêm...,J'étais très inquiète d'avoir choisi un hôtel ...,juillet 2006,A voyagé avec des amis
1139,2Travel,Singapore,sept. 2005,4.0,très bon rapport qualité/prix propre \n de l'h...,Nous sommes restés ici pour deux nuits et nous...,mai 2005,


In [6]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Meknès
================================================
Librairie  : DrissionPage
Cible      : Ibis Meknes - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g293735-d299682-Reviews-Ibis_Meknes-Meknes_Fez_Meknes.html"
MAX_PAGES  = None          # None = toutes les pages
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_meknes.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Meknès")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Meknès
   URL       : https://www.tripadvisor.fr/Hotel_Review-g293735-d299682-Reviews-Ibis_Meknes-Meknes_Fez_Meknes.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre toutes langues via URL

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 120

📄 Page 13
  → 1

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Hamza K,1 contribution,avr. 2026,5.0,Excellent,Hôtel bien situé au centre de la ville \nChamb...,avril 2026,A voyagé en solo
1,Aya L,1 contribution,août 2025,5.0,Superbe,Hôtel très propre a recommander service petit ...,août 2025,A voyagé en solo
2,Driss e,1 contribution,juil. 2025,1.0,Accueil désagréable hôtel en déclin,Je tiens à faire part de ma grande déception c...,juin 2025,A voyagé en famille
3,Rascal33,11 contributions,mai 2025,2.0,"Pas de Clim … avec 30° dans les chambres, mais...","Pas de Clim, et le personnel nous prend vraime...",mai 2025,A voyagé en couple
4,Alana V,1 contribution,oct. 2024,1.0,"Refusé notre séjour, couple nouvellement marié",Mon partenaire marocain et moi nous sommes mar...,septembre 2024,A voyagé en couple
...,...,...,...,...,...,...,...,...
412,intrepidtraveller39,Kent,juin 2007,5.0,Excellent rapport qualité/prix,Entre vos visites touristiques Rabat et Fez No...,juin 2007,A voyagé en couple
413,Alan1971Uk,UK,mai 2007,5.0,Encore un excellent Ibis,Un excellent rapport qualité/prix Ibis. Nous a...,mai 2007,A voyagé en couple
414,Al M,"Duenweg, Missouri",janv. 2007,5.0,Un super hôtel à \n de Mekhnès,"Cet hôtel, situé entre les parties anciennes q...",décembre 2006,A voyagé pour affaires
415,wheelo,"Sydney, Australie",août 2006,3.0,Demandez une chambre double avec vue sur la pi...,"Nous connaissions la chaîne Ibis, donc cet hôt...",août 2006,A voyagé en couple


In [7]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Marrakech Centre Gare
==============================================================
Librairie  : DrissionPage
Cible      : Ibis Marrakech Centre Gare - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g293734-d299643-Reviews-Ibis_Marrakech_Centre_Gare-Marrakech_Marrakech_Safi.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_marrakech.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Marrakech Centre Gare")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Marrakech Centre Gare
   URL       : https://www.tripadvisor.fr/Hotel_Review-g293734-d299643-Reviews-Ibis_Marrakech_Centre_Gare-Marrakech_Marrakech_Safi.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 av

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Jack,"Stockholm, Suède",avr. 2026,5.0,Les premières impressions comptent vraiment,J'ai eu un contrôle très tardif vers 2 heures ...,avril 2026,A voyagé en solo
1,Saida l,1 contribution,avr. 2026,5.0,Meilleure expérience,C'était vraiment une super expérience pour moi...,avril 2026,A voyagé avec des amis
2,Mouad N,1 contribution,avr. 2026,5.0,Excellent séjour à Marrakech,Séjour très agréable ! L’hôtel est bien situé ...,avril 2026,A voyagé en solo
3,Mohammad K,1 contribution,avr. 2026,5.0,Oke,Service bon \nPdj améliorée staff a l'ecoutr,avril 2026,A voyagé en solo
4,Journey66665812106,5 contributions,avr. 2026,5.0,Bon service .a rénové,Chambre okkkk besoin de rénovation quelque par...,avril 2026,A voyagé en solo
...,...,...,...,...,...,...,...,...
1546,Mark2211,Grand Cayman,juil. 2006,4.0,Très bel,C'est un bel hôtel. Les chambres sont simples ...,juin 2006,
1547,Steve C,Kent,avr. 2006,4.0,Bien mais vérifiez \n facture,"Un endroit agréable, avec des installations pr...",avril 2006,
1548,Trail35118,London,mars 2006,5.0,L'hôtel Ibis,Lors de mon second séjour à Marrakech Je suis ...,,
1549,sleepylagoons,Bedford UK,mars 2006,3.0,un bon \n de l'hôtel à prix réduit,Nous avons séjourné ici pendant deux nuits en ...,février 2006,


In [8]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis El Jadida
===================================================
Librairie  : DrissionPage
Cible      : Ibis El Jadida - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g298348-d572871-Reviews-Ibis_El_Jadida_Hotel-El_Jadida_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_eljadida.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis El Jadida")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis El Jadida
   URL       : https://www.tripadvisor.fr/Hotel_Review-g298348-d572871-Reviews-Ibis_El_Jadida_Hotel-El_Jadida_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre toutes langues via URL

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé 

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Fetah,"Casablanca, Maroc",avr. 2026,5.0,Exceptionnel,J'ai passé un excellent séjour à l'Ibis El Jad...,avril 2026,A voyagé en famille
1,Climber18126600285,1 contribution,avr. 2026,5.0,Séjour parfait du début à la fin,J'ai eu une excellente expérience à l'hôtel ib...,avril 2026,A voyagé avec des amis
2,Nouhaila L,1 contribution,avr. 2026,5.0,Bonne journée,J'ai passé une très belle expérience avec mes ...,avril 2026,A voyagé pour affaires
3,Rabiaa A,1 contribution,avr. 2026,5.0,Très bon service,J'ai passé une très bonne journée d'étude avec...,avril 2026,A voyagé pour affaires
4,Souaad M,"Casablanca, Maroc",avr. 2026,5.0,Barvo,Bon séjour merci beaucoup pour les équipe de h...,avril 2026,A voyagé en famille
...,...,...,...,...,...,...,...,...
704,coemgenus,"Londres, Royaume-Uni",janv. 2008,3.0,Un choix judicieux décevant par le personnel,En décembre 2007 L'hôtel était un vrai soulage...,décembre 2007,
705,hysopes,villemomble,févr. 2007,5.0,endrois sympatique meme en janvier,ayant passé plusieurs jours dans c'est hotel a...,janvier 2007,
706,stego,"Lisbonne, Portugal",sept. 2006,4.0,Bon rapport qualité/prix 3 étoiles. Logement s...,"Ma femme, mon fils de huit ans et moi avons pa...",août 2006,A voyagé en famille
707,KarenI,UK,août 2006,3.0,"Bel hôtel, ne vous attendez pas à un lit pour ...","L'hôtel était charmant, propre, bien décoré, l...",juillet 2006,A voyagé en famille


In [9]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Ouarzazate Centre
===========================================================
Librairie  : DrissionPage
Cible      : Ibis Ouarzazate Centre - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g304018-d602390-Reviews-Ibis_Ouarzazate_Centre-Ouarzazate_Draa_Tafilalet.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_ouarzazate.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Ouarzazate Centre")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Ouarzazate Centre
   URL       : https://www.tripadvisor.fr/Hotel_Review-g304018-d602390-Reviews-Ibis_Ouarzazate_Centre-Ouarzazate_Draa_Tafilalet.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre toutes langues via URL

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,BostonNix,"Cape Cod, Massachusetts",avr. 2026,2.0,Ok pour une nuit,Cet hôtel est idéalement situé mais est mal en...,avril 2026,A voyagé en couple
1,Anouar A,1 contribution,avr. 2026,5.0,Magnifique,Je suis très satisfait pour cet hôtel c'est ma...,avril 2026,A voyagé en famille
2,Gh T,1 contribution,avr. 2026,5.0,Jolie,Agréable séjour et magnifique jour et surtout ...,avril 2026,A voyagé en solo
3,Hayat H,1 contribution,avr. 2026,5.0,Magnifique,Hôtel magnifique je suis vraiment contente le ...,avril 2026,A voyagé pour affaires
4,Brahim B,1 contribution,avr. 2026,5.0,رائعة,تجربة رائعة حظيت بوقت رائع وممتع من وجهة \nنظر...,avril 2026,A voyagé en famille
...,...,...,...,...,...,...,...,...
813,AdrianaPerezPesce,Madrid,mars 2007,4.0,Gate to desert,"Nice place, clean, good pool and confortable.",mars 2007,A voyagé pour affaires
814,MikkiEast,"Rostock, Allemagne",déc. 2006,5.0,"meilleur prix, meilleure \n du service",C'est vraiment incroyable mais pour à ce prix-...,novembre 2006,A voyagé pour affaires
815,cger04,"Seattle, WA",nov. 2006,2.0,Drôle d'hôtel,C'est un hôtel assez bizarre. Il n'y a aucun p...,octobre 2006,A voyagé en famille
816,Wogo,"New York, NY",nov. 2006,5.0,Excellent rapport qualité/prix,J'étais un peu réticent à l'idée de réserver u...,novembre 2006,


In [11]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Marrakech Palmeraie
=============================================================
Librairie  : DrissionPage
Cible      : Ibis Marrakech Palmeraie - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g293734-d627810-Reviews-Ibis_Marrakech_Palmeraie-Marrakech_Marrakech_Safi.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_marrakech_palmeraie.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Marrakech Palmeraie")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Marrakech Palmeraie
   URL       : https://www.tripadvisor.fr/Hotel_Review-g293734-d627810-Reviews-Ibis_Marrakech_Palmeraie-Marrakech_Marrakech_Safi.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,ratonlaveur2014,"Quebec City, Quebec",avr. 2026,4.0,"Hôtel parfait pour relaxer, personnel hyper ge...",À l'exception d'une localisation quelque peu l...,avril 2026,A voyagé en couple
1,Trail65077464846,2 contributions,avr. 2026,2.0,hotel tres moyen,"Points forts :\nLe personnel est accueillant, ...",avril 2026,A voyagé en solo
2,Toucan62,"Villejuif, France",mars 2026,3.0,Bon marché mais refresh à faire,Hôtel vieillissant qui aurait vraiment besoin ...,mars 2026,A voyagé en famille
3,Laurent B,cesson sevigne,févr. 2026,5.0,UNE BONNE ADRESSE À MARRAKECH,Parfait sur toute la ligne... Accueil agréable...,février 2026,A voyagé en couple
4,M W,"Londres, Royaume-Uni",févr. 2026,3.0,Moyenne,Une étoile de moins pour l'emplacement et une ...,janvier 2026,A voyagé en solo
...,...,...,...,...,...,...,...,...
638,Tifette,Belgique,août 2007,4.0,hotel agréable et propre,"hotel propre et agréable, piscine bien entrete...",août 2007,
639,contempleuse,belgique,avr. 2007,4.0,pas mal mais loin du centre,"Hotel récent, très propre, belles chambres spa...",avril 2007,A voyagé en famille
640,redsilence1,"Maidstone, Royaume-Uni",févr. 2007,5.0,"En dehors de la ville, mais ça vaut le coup !",Un hôtel récent et à prix réduit de 2 km de la...,février 2007,A voyagé pour affaires
641,Frederic_G,"Gérone, Espagne",janv. 2007,4.0,Bueno y barato,Hotel nuevo (inaugurado al principio de Diciem...,décembre 2006,A voyagé en famille


In [3]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Casablanca City Center
================================================================
Librairie  : DrissionPage
Cible      : Ibis Casablanca City Center - TripAdvisor FR

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.fr/Hotel_Review-g293732-d1019409-Reviews-Ibis_Casablanca_City_Center-Casablanca_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_casablanca.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+sur", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)", txt)
    return m.group(1).strip() if m else ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele('css:a[aria-label="Page suivante"]', timeout=5)
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Casablanca City Center")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Casablanca City Center
   URL       : https://www.tripadvisor.fr/Hotel_Review-g293732-d1019409-Reviews-Ibis_Casablanca_City_Center-Casablanca_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Mostapha B,1 contribution,1 mai,5.0,Ibis Casablanca review. I am a google guide.,Me and my mother booked one night in Ibis Casa...,mai 2026,A voyagé en famille
1,Hind T,1 contribution,1 mai,5.0,Petit déjeuner délicieux et les chembre propre,Petit déjeuner parfait \nMerci Alia pour mesme...,mai 2026,A voyagé en famille
2,Casa C,1 contribution,avr. 2026,5.0,Le personnel très aimable . merci Asmaa,Une expérience exceptionnelle l'hôtel est très...,avril 2026,A voyagé pour affaires
3,Odyssey12126791652,1 contribution,avr. 2026,5.0,Le personnel est très aimable,Séjour est excellent.\nLes restaurants excelle...,avril 2026,A voyagé en famille
4,Vincente I,1 contribution,avr. 2026,5.0,Mon séjour parfait à ibis à casa city centre,Petit déjeuner \nMerci à tout touria pour les ...,avril 2026,A voyagé pour affaires
...,...,...,...,...,...,...,...,...
1471,ThePrinceFamily,Paris,mai 2008,3.0,Bien situé et correct mais pour combien de tem...,Nous avons été parmis les premiers clients de ...,mai 2008,A voyagé pour affaires
1472,The Wayfarer,"Londres, Royaume-Uni",mai 2008,4.0,Bon rapport qualité \n d'excellent rapport qua...,J'ai séjourné ici deux nuits (19e et 20 avril)...,avril 2008,
1473,Limatwice,"Île-de-France, France",mai 2008,5.0,le meilleur rapport qualité-prix à Casa!,"Très grand hôtel, tout neuf, standard Ibis hab...",mars 2008,A voyagé pour affaires
1474,Luca,"Lugano, Suisse",mars 2008,5.0,Tout simplement génial !,Super hôtel ! Tout neuf gratte-ciel (17. Le so...,mars 2008,


In [1]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Casa Sidi Maarouf
===========================================================
Librairie  : DrissionPage
Cible      : Ibis Casa Sidi Maarouf - TripAdvisor

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.com/Hotel_Review-g293732-d638968-Reviews-Ibis_Casa_Sidi_Maarouf-Casablanca_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_casa_sidi_maarouf.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+(sur|of)", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)|review\s+(?:written\s+)?(?:on\s+)?(.+)", txt, re.I)
    if m:
        return (m.group(1) or m.group(2) or "").strip()
    return ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")'
            ' or contains(.,"Accept")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        # .com utilise "Next page" en anglais
        btn = page.ele(
            'css:a[aria-label="Page suivante"], a[aria-label="Next page"]',
            timeout=5
        )
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Casa Sidi Maarouf")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Casa Sidi Maarouf
   URL       : https://www.tripadvisor.com/Hotel_Review-g293732-d638968-Reviews-Ibis_Casa_Sidi_Maarouf-Casablanca_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre toutes langues via URL

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  t

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Fetah,"Casablanca, Morocco",Apr 2026,5.0,Excellent,Hotel perfectly suited for business travel in ...,April 2026,Traveled on business
1,DRM,4 contributions,Apr 2026,5.0,Very good stay at the ibis Sidi Maarouf.,"The hotel enjoys a convenient location, with s...",April 2026,Traveled on business
2,Malhi A,2 contributions,Apr 2026,5.0,Exceptional,Exceptional stay at the Ibis Casablanca Sidi M...,April 2026,Traveled on business
3,Khadija M,14 contributions,Apr 2026,5.0,Service recovery,"After a difficult start to my stay, I woke up ...",April 2026,Traveled on business
4,Andrea77R,"Cavriago, Italy",Nov 2025,3.0,The classic franchise.... average,A bit noisy area of the city due to the proxim...,October 2025,Traveled on business
...,...,...,...,...,...,...,...,...
531,renaudaja,corse,Mar 2010,3.0,ouai pourquoi pas?!,alors juste la pour le WE\n1 er point hotel as...,February 2010,Traveled as a couple
532,TabarelliP,"Verona, Italy",Mar 2010,4.0,ibis,il miglior compromesso per ovunque si vada all...,August 2009,Traveled with friends
533,Hot-el-Test-4U,"Munich, Germany",Nov 2009,5.0,Preis / Leistung in Ordnung / ruhig,Positiv:\nKartenzahlung war problemlos möglich...,November 2009,Traveled on business
534,le D,"Paris, France",Nov 2009,3.0,de l'Ibis sans surprise,"IBIS standard, récent avec déco moderne, situé...",October 2009,Traveled on business


In [3]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Tanger City Center
============================================================
Librairie  : DrissionPage
Cible      : Ibis Tanger City Center - TripAdvisor

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.com/Hotel_Review-g293737-d2026212-Reviews-Ibis_Tanger_City_Center-Tangier_Tanger_Tetouan_Al_Hoceima.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_tanger.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+(sur|of)", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)|review\s+(?:written\s+)?(?:on\s+)?(.+)", txt, re.I)
    if m:
        return (m.group(1) or m.group(2) or "").strip()
    return ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")'
            ' or contains(.,"Accept")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele(
            'css:a[aria-label="Page suivante"], a[aria-label="Next page"]',
            timeout=5
        )
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Tanger City Center")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Tanger City Center
   URL       : https://www.tripadvisor.com/Hotel_Review-g293737-d2026212-Reviews-Ibis_Tanger_City_Center-Tangier_Tanger_Tetouan_Al_Hoceima.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,412margaret,"Stabroek, Belgium",Nov 2025,4.0,A perfect base for visiting Tangier,"As always, I was welcomed back to the Ibis Hot...",November 2025,Traveled solo
1,Ashraf G,1 contribution,Nov 2025,1.0,Such a shame but terrible,Tbh the only plus to this hotel is the locatio...,October 2025,Traveled with family
2,PAUL V,"Chambery, France",Oct 2025,2.0,Awakened at 2.00 in the morning!!,Welcoming very friendly. Functional room (quic...,October 2025,Traveled on business
3,Daniela M,1 contribution,Oct 2025,1.0,Terrible Experience!,Terrible experience — the room smells horribly...,October 2025,Traveled as a couple
4,Talabarder,"Le Havre, France",Sep 2025,1.0,Noisy,2 days 2 bedrooms. The rooms are noisy you can...,September 2025,Traveled on business
...,...,...,...,...,...,...,...,...
1029,retired4ever,Essaouira,Apr 2011,4.0,Excellent compromis pour une pause en remontan...,Hôtel récent dans un quartier en construction ...,April 2011,Traveled as a couple
1030,ALBA-NOHELIA,"Tangier, Morocco",Apr 2011,1.0,muy mal trato y se meten a los cuartos,"Estuve hosedada ahí por casi 20 días, en el tr...",March 2011,Traveled on business
1031,jayjay95,"Cergy, France",Apr 2011,4.0,tres bon hotel,J'ai vraiment aimé cet hotel fraichement ouver...,March 2011,Traveled as a couple
1032,cymruDarija,"Cardiff, United Kingdom",Mar 2011,5.0,Very Nice Hotel: Simple but quite generous,"I have to give this hotel 5 stars , as Two nig...",March 2011,


In [4]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Casablanca Nearshore
====================================================
Librairie  : DrissionPage
Cible      : Ibis Casablanca - TripAdvisor

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.com/Hotel_Review-g293732-d6467850-Reviews-Ibis_Casablanca-Casablanca_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_casablanca_nearshore.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+(sur|of)", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)|review\s+(?:written\s+)?(?:on\s+)?(.+)", txt, re.I)
    if m:
        return (m.group(1) or m.group(2) or "").strip()
    return ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")'
            ' or contains(.,"Accept")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele(
            'css:a[aria-label="Page suivante"], a[aria-label="Next page"]',
            timeout=5
        )
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Casablanca")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Casablanca
   URL       : https://www.tripadvisor.com/Hotel_Review-g293732-d6467850-Reviews-Ibis_Casablanca-Casablanca_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  total cum

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,BOUZID H,1 contribution,May 2,5.0,Excellent,Merci à tous pour votre hospitalité et votre a...,May 2026,Traveled on business
1,Badr F,1 contribution,Apr 2026,5.0,Conformable,The staff are very kind and rooms are clean i ...,April 2026,Traveled on business
2,Fatima H,2 contributions,Apr 2026,5.0,Très beau,"L'accueil chaleureux,les chambres moderne et p...",April 2026,Traveled on business
3,Saad H,1 contribution,Apr 2026,5.0,Bon prix,Hotels magnifique les chambte propres merci a ...,April 2026,Traveled on business
4,Companion67039245107,1 contribution,Apr 2026,5.0,Enjoy your stay,Hotels calm the beautiful area the rooms are v...,April 2026,Traveled on business
...,...,...,...,...,...,...,...,...
696,Farid L,"Algiers, Algeria",Jun 2014,4.0,Hôtel récemment ouvert,Ouvert depuis moins de 2 mois donc tout est ne...,June 2014,Traveled on business
697,DCRAntibes,"Antibes, France",Jun 2014,4.0,Trés calme et bien placé,Ce nouvel hôtel est ouvert depuis Mai 14 au se...,June 2014,Traveled on business
698,nick3317,"Enfield, United Kingdom",Jun 2014,4.0,friendly and still intact,in a fully new area of university the hotel is...,June 2014,Traveled on business
699,Miguel M,Alcoy,May 2014,4.0,Nuevo hotel muy cómodo en Casablanca,Estuve tan solo una semana tras su inaguración...,May 2014,Traveled on business


In [5]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Mohammedia
====================================================
Librairie  : DrissionPage
Cible      : Ibis Mohammedia - TripAdvisor

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.com/Hotel_Review-g2095768-d12704008-Reviews-Ibis_Mohammedia-Mohammedia_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_mohammedia.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+(sur|of)", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)|review\s+(?:written\s+)?(?:on\s+)?(.+)", txt, re.I)
    if m:
        return (m.group(1) or m.group(2) or "").strip()
    return ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")'
            ' or contains(.,"Accept")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele(
            'css:a[aria-label="Page suivante"], a[aria-label="Next page"]',
            timeout=5
        )
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Mohammedia")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Mohammedia
   URL       : https://www.tripadvisor.com/Hotel_Review-g2095768-d12704008-Reviews-Ibis_Mohammedia-Mohammedia_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ⚠ Filtre langue : 
该元素没有位置及大小。
版本: 4.1.1.2

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  t

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Rene V,1 contribution,Apr 2026,5.0,Very good hotel and staff,"Room was nice clean,zahira was very friendly.\...",April 2026,Traveled on business
1,Rachid R,1 contribution,Apr 2026,5.0,Clean hotel,Beautiful experience during my stay at the ibi...,April 2026,Traveled on business
2,Maman M,2 contributions,Apr 2026,5.0,Great stay,"Amazing hôtel، great and helpfull staff, Well ...",April 2026,Traveled with friends
3,Fjhfjfhr J,2 contributions,Apr 2026,5.0,Nice hotel,"Good location, very friendly staff, clean and ...",April 2026,Traveled on business
4,Nadia N,2 contributions,Apr 2026,5.0,Very satisfied,"For the second time at the ibis Mohammedia, I ...",April 2026,Traveled solo
...,...,...,...,...,...,...,...,...
157,bryann616,1 contribution,Oct 2017,5.0,Très beau hotel,Après 6 semaines passée dans ce flambant neuve...,October 2017,Traveled on business
158,Youssef07,2 contributions,Oct 2017,5.0,Merveilleux,"Flambants neuves, les parties communes de l'hô...",October 2017,Traveled on business
159,Zineb A,1 contribution,Oct 2017,5.0,excellent,Après mon passage à l’hôtel ibis à mohamedia j...,September 2017,Traveled on business
160,Marco C.,"Perugia, Italy",Oct 2017,4.0,"Ottimo Hotel a Mohammedia, nuovo",Sono stato in questo hotel per motivi di lavor...,October 2017,Traveled on business


In [7]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Casa Voyageurs
========================================================
Librairie  : DrissionPage
Cible      : Ibis Casa Voyageurs - TripAdvisor

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.com/Hotel_Review-g293732-d12842720-Reviews-Ibis_Casa_Voyageurs-Casablanca_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_casa_voyageurs.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+(sur|of)", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)|review\s+(?:written\s+)?(?:on\s+)?(.+)", txt, re.I)
    if m:
        return (m.group(1) or m.group(2) or "").strip()
    return ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")'
            ' or contains(.,"Accept")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele(
            'css:a[aria-label="Page suivante"], a[aria-label="Next page"]',
            timeout=5
        )
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Casa Voyageurs")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Casa Voyageurs
   URL       : https://www.tripadvisor.com/Hotel_Review-g293732-d12842720-Reviews-Ibis_Casa_Voyageurs-Casablanca_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 10 carte(s)
  ✓ 10 avis  |  

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Manar B,1 contribution,Feb 2026,1.0,Disappointment at breakfast,"Hello,\n\nI am disappointed by the attitude of...",February 2026,Traveled on business
1,jackietraveller,"Bakewell, United Kingdom",Jan 2026,5.0,Ticks all the boxes,Lovely clean hotel with nice staff . Ticks all...,January 2026,Traveled as a couple
2,moju72,Delhi,Dec 2025,2.0,Stuffy room,Stayed here one night. Convenient location for...,December 2025,Traveled solo
3,Hasna J,4 contributions,Dec 2025,5.0,Unforgettable,Very nice experience.thank you to All staff. L...,December 2025,Traveled with friends
4,Denis F,2 contributions,Nov 2025,4.0,Will stay again,"Staff is very pleasant. Rooms are clean, short...",November 2025,Traveled solo
...,...,...,...,...,...,...,...,...
285,FOUAD-RAK,"Agadir, Morocco",Oct 2017,5.0,"Hôtel très bien situé, personnel professionnel","Très bone accueil, Rapport qualité/prix raison...",October 2017,Traveled on business
286,Rene B,"Agadir, Morocco",Oct 2017,4.0,SEJOUR PLAISANT,La direction doit être plus flexible sur les m...,October 2017,Traveled on business
287,Francis Gerald,"Summerside, Canada",Oct 2017,1.0,Terrible Receptionist,We’re making this review just a few minutes af...,October 2017,Traveled as a couple
288,Mohamed F,"Utrecht, The Netherlands",Oct 2017,5.0,The happy place to be,"Dans cet hôtel, je me sens comme chez moi grâc...",October 2017,Traveled solo


In [8]:
"""
TripAdvisor Hotel Reviews Scraper — Ibis Abdelmoumen Casa
==========================================================
Librairie  : DrissionPage
Cible      : Ibis Abdelmoumen Casa - TripAdvisor

Installation :
    pip install DrissionPage pandas

Usage Jupyter : colle dans une cellule et exécute
"""

import time, random, re
from dataclasses import dataclass, asdict
from typing import Optional
import pandas as pd
from DrissionPage import ChromiumPage, ChromiumOptions


# ─────────────────────────────────────────
#  Configuration
# ─────────────────────────────────────────
URL = "https://www.tripadvisor.com/Hotel_Review-g293732-d20036182-Reviews-Ibis_Abdelmoumen_Casa-Casablanca_Casablanca_Settat.html"
MAX_PAGES  = None
HEADLESS   = False
OUTPUT_CSV = "tripadvisor_reviews_abdelmoumen.csv"
DELAY_MIN, DELAY_MAX = 2.5, 5.0


# ─────────────────────────────────────────
#  Modèle de données
# ─────────────────────────────────────────
@dataclass
class Review:
    auteur:           str            = ""
    contributions:    str            = ""
    date_publication: str            = ""
    note:             Optional[float] = None
    titre:            str            = ""
    texte:            str            = ""
    date_sejour:      str            = ""
    type_voyage:      str            = ""


# ─────────────────────────────────────────
#  Helpers
# ─────────────────────────────────────────
def rdelay():
    time.sleep(random.uniform(DELAY_MIN, DELAY_MAX))

def parse_note(txt: str) -> Optional[float]:
    m = re.search(r"(\d[\d,\.]*)\s+(sur|of)", txt)
    return float(m.group(1).replace(",", ".")) if m else None

def parse_date_pub(txt: str) -> str:
    m = re.search(r"avis\s+en\s+(.+)|review\s+(?:written\s+)?(?:on\s+)?(.+)", txt, re.I)
    if m:
        return (m.group(1) or m.group(2) or "").strip()
    return ""


# ─────────────────────────────────────────
#  Driver
# ─────────────────────────────────────────
def build_driver() -> ChromiumPage:
    co = ChromiumOptions()
    if HEADLESS:
        co.headless()
    co.set_argument("--no-sandbox")
    co.set_argument("--disable-dev-shm-usage")
    co.set_argument("--lang=fr-FR")
    co.set_argument("--window-size=1400,900")
    co.set_argument("--disable-blink-features=AutomationControlled")
    page = ChromiumPage(addr_or_opts=co)
    print("  ✓ Navigateur démarré")
    return page


# ─────────────────────────────────────────
#  Cookies
# ─────────────────────────────────────────
def accept_cookies(page):
    try:
        btn = page.ele(
            'xpath://button[contains(.,"Tout accepter") or contains(.,"Accepter")'
            ' or contains(.,"Accept")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Cookies acceptés")
            time.sleep(1.5)
        else:
            print("  ℹ Pas de bannière cookies")
    except Exception:
        print("  ℹ Pas de bannière cookies")


# ─────────────────────────────────────────
#  Sélectionner "Toutes les langues"
# ─────────────────────────────────────────
def select_all_languages(page):
    try:
        btn = page.ele(
            'xpath://*[contains(text(),"Toutes les langues") or contains(text(),"All languages")'
            ' or contains(text(),"Todas los idiomas")]',
            timeout=6
        )
        if btn:
            btn.click()
            print("  ✓ Filtre 'Toutes les langues' activé")
            time.sleep(2.5)
        else:
            current_url = page.url
            if "filterLang" not in current_url:
                sep = "&" if "?" in current_url else "?"
                page.get(current_url + sep + "filterLang=ALL")
                print("  ✓ Filtre toutes langues via URL")
                time.sleep(2.5)
            else:
                print("  ℹ Filtre langue non trouvé")
    except Exception as e:
        print(f"  ⚠ Filtre langue : {e}")


# ─────────────────────────────────────────
#  Expand boutons "Plus"
# ─────────────────────────────────────────
def expand_reviews(page):
    for btn in page.eles('css:button.UikNM'):
        try:
            btn.click()
            time.sleep(0.3)
        except Exception:
            pass


# ─────────────────────────────────────────
#  Parser une page
# ─────────────────────────────────────────
def parse_page(page) -> list:
    reviews = []
    cards = page.eles('css:div.ryPjd')
    print(f"  → {len(cards)} carte(s)")

    for card in cards:
        r = Review()

        # Auteur
        auteur_el = card.ele('css:span.OgHoE', timeout=1)
        r.auteur = auteur_el.text.strip() if auteur_el else ""

        # Date de publication
        pub_el = card.ele('css:div.VImYz.AWdfh', timeout=1)
        if pub_el:
            r.date_publication = parse_date_pub(pub_el.text)

        # Contributions
        contrib_el = card.ele('css:span.qVkLn', timeout=1)
        r.contributions = contrib_el.text.strip() if contrib_el else ""

        # Note
        note_el = card.ele('css:svg.evwcZ', timeout=1)
        if note_el:
            r.note = parse_note(note_el.text)

        # Titre
        titre_container = card.ele('css:div._a', timeout=1)
        if titre_container:
            titre_span = titre_container.ele('css:span.OgHoE', timeout=1)
            r.titre = titre_span.text.strip() if titre_span else ""
        if not r.titre:
            spans = card.eles('css:span.OgHoE')
            if len(spans) >= 2:
                r.titre = spans[-1].text.strip()

        # Texte de l'avis
        texte_el = card.ele('css:div.fIrGe', timeout=1)
        r.texte = texte_el.text.strip() if texte_el else ""

        # Date séjour & Type de voyage
        details_el = card.ele('css:div.ERnxS', timeout=1)
        if details_el:
            xenvs = details_el.eles('css:span.xENVe')
            if len(xenvs) >= 1:
                r.date_sejour = xenvs[0].text.strip()
            if len(xenvs) >= 2:
                r.type_voyage = xenvs[1].text.strip()

        if not r.auteur and not r.titre:
            continue

        reviews.append(r)

    return reviews


# ─────────────────────────────────────────
#  Pagination
# ─────────────────────────────────────────
def next_page(page) -> bool:
    try:
        btn = page.ele(
            'css:a[aria-label="Page suivante"], a[aria-label="Next page"]',
            timeout=5
        )
        if not btn:
            return False
        try:
            btn.scroll.into_view()
        except Exception:
            pass
        time.sleep(0.6)
        btn.click()
        rdelay()
        return True
    except Exception:
        return False


# ─────────────────────────────────────────
#  Sauvegarde CSV
# ─────────────────────────────────────────
def save(all_reviews: list):
    df = pd.DataFrame([asdict(r) for r in all_reviews])
    df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
    print(f"\n✅ CSV  → {OUTPUT_CSV}")
    print(f"\n{'='*50}")
    print(f"  Total avis extraits : {len(all_reviews)}")
    notes = [r.note for r in all_reviews if r.note is not None]
    if notes:
        print(f"  Note moyenne        : {sum(notes)/len(notes):.2f} / 5")
    print(f"{'='*50}")
    return df


# ─────────────────────────────────────────
#  Main
# ─────────────────────────────────────────
def main():
    print("🚀 TripAdvisor Scraper — Ibis Abdelmoumen Casa")
    print(f"   URL       : {URL}")
    print(f"   Max pages : {MAX_PAGES or 'toutes'}\n")

    page = build_driver()
    all_reviews = []

    try:
        page.get(URL)
        rdelay()
        accept_cookies(page)
        select_all_languages(page)  # ✅ toutes les langues

        pg = 1
        while True:
            print(f"\n📄 Page {pg}")

            page.scroll.to_top()
            for _ in range(12):
                page.scroll.down(700)
                time.sleep(0.2)

            expand_reviews(page)
            time.sleep(1.2)

            reviews = parse_page(page)
            all_reviews.extend(reviews)
            print(f"  ✓ {len(reviews)} avis  |  total cumulé : {len(all_reviews)}")

            if MAX_PAGES and pg >= MAX_PAGES:
                print("\n  ⏹ Limite de pages atteinte.")
                break

            if not next_page(page):
                print("\n  ⏹ Dernière page — scraping terminé.")
                break

            pg += 1
            rdelay()

    except KeyboardInterrupt:
        print("\n⚠ Arrêté manuellement — sauvegarde en cours...")
    except Exception as e:
        import traceback
        print(f"\n❌ Erreur : {e}")
        traceback.print_exc()
    finally:
        page.quit()

    if all_reviews:
        return save(all_reviews)
    else:
        print("\n⚠ Aucun avis extrait.")
        return None


main()

🚀 TripAdvisor Scraper — Ibis Abdelmoumen Casa
   URL       : https://www.tripadvisor.com/Hotel_Review-g293732-d20036182-Reviews-Ibis_Abdelmoumen_Casa-Casablanca_Casablanca_Settat.html
   Max pages : toutes

  ✓ Navigateur démarré
  ℹ Pas de bannière cookies
  ✓ Filtre 'Toutes les langues' activé

📄 Page 1
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 10

📄 Page 2
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 20

📄 Page 3
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 30

📄 Page 4
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 40

📄 Page 5
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 50

📄 Page 6
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 60

📄 Page 7
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 70

📄 Page 8
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 80

📄 Page 9
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 90

📄 Page 10
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 100

📄 Page 11
  → 10 carte(s)
  ✓ 10 avis  |  total cumulé : 110

📄 Page 12
  → 8 carte(s)
  ✓ 8 avis  |

,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage
0,Luis Antonio B,"Rio de Janeiro, RJ",Dec 2025,1.0,202511 - Terrible lodging in Casablanca,The standard room of the Ibis is larger than e...,November 2025,Traveled as a couple
1,Atif,"London, United Kingdom",Oct 2025,3.0,Budget hotel good value,The hotel was located in a very busy high stre...,October 2025,Traveled with family
2,Paradise00617377813,2 contributions,Oct 2025,5.0,They spare no effort for your comfort without ...,Perfect for my needs. Salma from quality contr...,October 2025,Traveled solo
3,pascal136,"Perpignan, France",Aug 2025,5.0,What we expect for ibis.,Good Price quality ratio. Staff attentive and ...,August 2025,Traveled with friends
4,khalil Z,2 contributions,Aug 2025,5.0,Perfect Hotel,I would like to thank you sincerely for the wa...,August 2025,
...,...,...,...,...,...,...,...,...
113,Djawad B,1 contribution,Mar 2020,5.0,Séjour agréable a recommander,Un cour séjour .\nJ’ai a passé un très beau sé...,March 2020,Traveled on business
114,Sarah S,1 contribution,Mar 2020,5.0,Parfaitement parfait,J'ai passé sincèrement un séjour inoubliable.....,March 2020,Traveled on business
115,Loubna E,6 contributions,Mar 2020,5.0,Weekend,"Parfait,agréable personnels attentifs et préve...",March 2020,Traveled as a couple
116,Riyad 777,4 contributions,Mar 2020,5.0,Satisfait,"Super hôtel ! Le personnel est au top, ils son...",February 2020,Traveled with family


In [9]:
"""
Combinaison de tous les CSV TripAdvisor Ibis Maroc
===================================================
Ajoute les colonnes : hotel_name, hotel_city, source
Output : tripadvisor_reviews_ALL.csv
"""

import pandas as pd
import os

# ─────────────────────────────────────────
#  Mapping fichier → (hotel_name, hotel_city)
# ─────────────────────────────────────────
FILES = {
    "tripadvisor_reviews_abdelmoumen.csv":        ("Ibis Abdelmoumen Casa",          "Casablanca"),
    "tripadvisor_reviews_casa_sidi_maarouf.csv":  ("Ibis Casa Sidi Maarouf",         "Casablanca"),
    "tripadvisor_reviews_casa_voyageurs.csv":     ("Ibis Casa Voyageurs",            "Casablanca"),
    "tripadvisor_reviews_casablanca_nearshore.csv":("Ibis Casablanca Nearshore",     "Casablanca"),
    "tripadvisor_reviews_casablanca.csv":         ("Ibis Casablanca City Center",    "Casablanca"),
    "tripadvisor_reviews_eljadida.csv":           ("Ibis El Jadida",                 "El Jadida"),
    "tripadvisor_reviews_fes.csv":                ("Ibis Fès",                       "Fès"),
    "tripadvisor_reviews_marrakech_palmeraie.csv":("Ibis Marrakech Palmeraie",       "Marrakech"),
    "tripadvisor_reviews_marrakech.csv":          ("Ibis Marrakech Centre Gare",     "Marrakech"),
    "tripadvisor_reviews_meknes.csv":             ("Ibis Meknès",                    "Meknès"),
    "tripadvisor_reviews_mohammedia.csv":         ("Ibis Mohammedia",                "Mohammedia"),
    "tripadvisor_reviews_ouarzazate.csv":         ("Ibis Ouarzazate Centre",         "Ouarzazate"),
    "tripadvisor_reviews_oujda.csv":              ("Ibis Oujda",                     "Oujda"),
    "tripadvisor_reviews_Rabat_agdal.csv":        ("Ibis Rabat Agdal",               "Rabat"),
    "tripadvisor_reviews_tanger.csv":             ("Ibis Tanger City Center",        "Tanger"),
}

SOURCE = "tripadvisor.com"

# ─────────────────────────────────────────
#  Combinaison
# ─────────────────────────────────────────
dfs = []
missing = []

for filename, (hotel_name, hotel_city) in FILES.items():
    if os.path.exists(filename):
        df = pd.read_csv(filename, encoding="utf-8-sig")
        df.insert(0, "hotel_name", hotel_name)
        df.insert(1, "hotel_city", hotel_city)
        df["source"] = SOURCE
        dfs.append(df)
        print(f"  ✓ {filename:<50} → {len(df):>4} avis")
    else:
        missing.append(filename)
        print(f"  ✗ {filename} — INTROUVABLE")

if not dfs:
    print("\n❌ Aucun fichier trouvé. Lance ce script dans le même dossier que les CSV.")
else:
    combined = pd.concat(dfs, ignore_index=True)
    combined.to_csv("avis_total_tripadvisor.csv", index=False, encoding="utf-8-sig")

    print(f"\n{'='*55}")
    print(f"  Fichiers combinés   : {len(dfs)}")
    if missing:
        print(f"  Fichiers manquants  : {len(missing)}")
    print(f"  Total avis          : {len(combined)}")
    print(f"  Colonnes            : {list(combined.columns)}")
    print(f"\n  Avis par hôtel :")
    for hotel, count in combined.groupby("hotel_name").size().items():
        print(f"    {hotel:<40} {count:>4} avis")
    print(f"\n✅ Fichier final → avis_total_tripadvisor.csv")
    print(f"{'='*55}")

  ✓ tripadvisor_reviews_abdelmoumen.csv                →  118 avis
  ✓ tripadvisor_reviews_casa_sidi_maarouf.csv          →  536 avis
  ✓ tripadvisor_reviews_casa_voyageurs.csv             →  290 avis
  ✓ tripadvisor_reviews_casablanca_nearshore.csv       →  701 avis
  ✓ tripadvisor_reviews_casablanca.csv                 → 1476 avis
  ✓ tripadvisor_reviews_eljadida.csv                   →  709 avis
  ✓ tripadvisor_reviews_fes.csv                        → 1141 avis
  ✓ tripadvisor_reviews_marrakech_palmeraie.csv        →  643 avis
  ✓ tripadvisor_reviews_marrakech.csv                  → 1551 avis
  ✓ tripadvisor_reviews_meknes.csv                     →  417 avis
  ✓ tripadvisor_reviews_mohammedia.csv                 →  162 avis
  ✓ tripadvisor_reviews_ouarzazate.csv                 →  818 avis
  ✓ tripadvisor_reviews_oujda.csv                      →  273 avis
  ✓ tripadvisor_reviews_Rabat_agdal.csv                →  672 avis
  ✓ tripadvisor_reviews_tanger.csv                     → 1034 

In [10]:
"""
Détection de langue des avis TripAdvisor
==========================================
Librairie : langdetect
Input     : avis_total_tripadvisor.csv
Output    : avis_total_tripadvisor.csv (avec colonne 'langue' ajoutée)

Installation :
    pip install langdetect
"""

import pandas as pd
from langdetect import detect, DetectorFactory
from langdetect.lang_detect_exception import LangDetectException

# Rendre la détection déterministe (résultats stables)
DetectorFactory.seed = 42

# ─────────────────────────────────────────
#  Chargement
# ─────────────────────────────────────────
INPUT_FILE  = "avis_total_tripadvisor.csv"
OUTPUT_FILE = "avis_total_tripadvisor.csv"

print(f"📂 Chargement de {INPUT_FILE}...")
df = pd.read_csv(INPUT_FILE, encoding="utf-8-sig")
print(f"  ✓ {len(df)} avis chargés\n")


# ─────────────────────────────────────────
#  Détection de langue
# ─────────────────────────────────────────
# Codes langue → noms complets
LANG_NAMES = {
    "fr": "Français",
    "en": "Anglais",
    "ar": "Arabe",
    "es": "Espagnol",
    "de": "Allemand",
    "it": "Italien",
    "pt": "Portugais",
    "nl": "Néerlandais",
    "ru": "Russe",
    "zh-cn": "Chinois",
    "ja": "Japonais",
    "ko": "Coréen",
    "tr": "Turc",
    "pl": "Polonais",
}

def detect_language(text: str) -> str:
    """Détecte la langue d'un texte. Retourne 'Inconnu' si impossible."""
    if not isinstance(text, str) or len(text.strip()) < 5:
        return "Inconnu"
    try:
        code = detect(text)
        return LANG_NAMES.get(code, code.upper())
    except LangDetectException:
        return "Inconnu"


print("🔍 Détection des langues en cours...")
total = len(df)

langues = []
for i, texte in enumerate(df["texte"], 1):
    langues.append(detect_language(texte))
    if i % 100 == 0 or i == total:
        print(f"  {i}/{total} avis traités...", end="\r")

df["langue"] = langues
print(f"\n  ✓ Détection terminée\n")


# ─────────────────────────────────────────
#  Sauvegarde
# ─────────────────────────────────────────
df.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"✅ Fichier sauvegardé → {OUTPUT_FILE}")


# ─────────────────────────────────────────
#  Résumé
# ─────────────────────────────────────────
print(f"\n{'='*45}")
print(f"  Répartition des langues détectées :")
print(f"{'='*45}")
lang_counts = df["langue"].value_counts()
for lang, count in lang_counts.items():
    pct = count / total * 100
    print(f"  {lang:<20} {count:>5} avis  ({pct:.1f}%)")
print(f"{'='*45}")
print(f"  TOTAL : {total} avis")

📂 Chargement de avis_total_tripadvisor.csv...
  ✓ 10541 avis chargés

🔍 Détection des langues en cours...
  10541/10541 avis traités...
  ✓ Détection terminée

✅ Fichier sauvegardé → avis_total_tripadvisor.csv

  Répartition des langues détectées :
  Français              6935 avis  (65.8%)
  Anglais               2066 avis  (19.6%)
  Espagnol               448 avis  (4.3%)
  Italien                389 avis  (3.7%)
  Allemand               205 avis  (1.9%)
  Portugais              195 avis  (1.8%)
  Néerlandais             86 avis  (0.8%)
  Russe                   46 avis  (0.4%)
  Arabe                   45 avis  (0.4%)
  Japonais                43 avis  (0.4%)
  Turc                    27 avis  (0.3%)
  Polonais                14 avis  (0.1%)
  Coréen                  13 avis  (0.1%)
  Chinois                  9 avis  (0.1%)
  ZH-TW                    7 avis  (0.1%)
  DA                       4 avis  (0.0%)
  SV                       3 avis  (0.0%)
  Inconnu                  1 avis  

In [12]:
import pandas as pd 
df = pd.read_csv("avis_total_tripadvisor.csv")
df.head()

,hotel_name,hotel_city,auteur,contributions,date_publication,note,titre,texte,date_sejour,type_voyage,source,langue
0,Ibis Abdelmoumen Casa,Casablanca,Luis Antonio B,"Rio de Janeiro, RJ",Dec 2025,1.0,202511 - Terrible lodging in Casablanca,The standard room of the Ibis is larger than e...,November 2025,Traveled as a couple,tripadvisor.com,Anglais
1,Ibis Abdelmoumen Casa,Casablanca,Atif,"London, United Kingdom",Oct 2025,3.0,Budget hotel good value,The hotel was located in a very busy high stre...,October 2025,Traveled with family,tripadvisor.com,Anglais
2,Ibis Abdelmoumen Casa,Casablanca,Paradise00617377813,2 contributions,Oct 2025,5.0,They spare no effort for your comfort without ...,Perfect for my needs. Salma from quality contr...,October 2025,Traveled solo,tripadvisor.com,Anglais
3,Ibis Abdelmoumen Casa,Casablanca,pascal136,"Perpignan, France",Aug 2025,5.0,What we expect for ibis.,Good Price quality ratio. Staff attentive and ...,August 2025,Traveled with friends,tripadvisor.com,Anglais
4,Ibis Abdelmoumen Casa,Casablanca,khalil Z,2 contributions,Aug 2025,5.0,Perfect Hotel,I would like to thank you sincerely for the wa...,August 2025,NaN,tripadvisor.com,Anglais
